# Amazon Entity Resolution — V7 CPU and GPU Kaggle notebook

This is a self-contained Kaggle notebook for the complete V7 workflow. It embeds the project scripts, so the new Kaggle account does **not** need GitHub access or a GitHub token. Attach the official challenge dataset with **Add Input**.

The notebook offers two model paths while keeping the same V7 candidate generation, 60 pair features, entity-level validation, macro F0.5 threshold sweep, full test inference, and output validation:

- `catboost_gpu`: CPU candidate/feature generation, CatBoost model fitting on both T4 GPUs when available.
- `lightgbm_cpu`: the existing V7 LightGBM CPU baseline.

Set `BACKEND` in the configuration cell and run the notebook top to bottom. Only the selected training path runs. GPU acceleration applies to the model fit; the blocking and pair-feature generation remain CPU work and may still be the longest stage. The full source tables are used; only easy negative *pairs* are sampled by the existing V7 training builder.

In [ ]:
from pathlib import Path
from io import BytesIO
import base64, zipfile

REPO_DIR = Path('/kaggle/working/AMAZON-ML-CHALLENGE')
REPO_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_B64 = """..."""
bundle = base64.b64decode(BUNDLE_B64)
with zipfile.ZipFile(BytesIO(bundle)) as zf:
    zf.extractall(REPO_DIR)
print('Embedded project code unpacked to:', REPO_DIR)
print('Included scripts:', *[p.relative_to(REPO_DIR) for p in REPO_DIR.rglob('*.py')], sep='\n  ')


In [ ]:
import subprocess, sys

# Keep Kaggle's preinstalled NumPy/SciPy stack intact. CatBoost's Linux wheel
# includes CUDA support; no LightGBM source compilation is needed.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'lightgbm==4.7.0', 'rapidfuzz==3.14.3', 'Unidecode==1.4.0', 'catboost==1.2.10'
], check=True)
import numpy, scipy, lightgbm, rapidfuzz, unidecode, catboost
print('Runtime:', 'numpy', numpy.__version__, '| lightgbm', lightgbm.__version__, '| catboost', catboost.__version__)


In [ ]:
from pathlib import Path, PurePosixPath
import zipfile

INPUT_ROOT = Path('/kaggle/input')
TRAIN_REQUIRED = {'train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv', 'train_ground_truth.tsv'}
TEST_REQUIRED = {'test_source1.tsv', 'test_source2.tsv', 'test_source3.tsv'}

def find_attached_dataset():
    found = []
    if INPUT_ROOT.exists():
        for source1 in INPUT_ROOT.rglob('train_source1.tsv'):
            train_dir = source1.parent
            test_dir = train_dir.parent / 'test'
            train_files = {p.name for p in train_dir.iterdir() if p.is_file()}
            test_files = {p.name for p in test_dir.iterdir() if p.is_file()} if test_dir.is_dir() else set()
            if TRAIN_REQUIRED.issubset(train_files) and TEST_REQUIRED.issubset(test_files):
                found.append((train_dir, test_dir))
    return found

def safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            rel = PurePosixPath(member.filename)
            if rel.is_absolute() or '..' in rel.parts:
                raise RuntimeError(f'Unsafe path in attached archive: {member.filename}')
            target = (destination / Path(*rel.parts)).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f'Archive member escapes extraction folder: {member.filename}')
        zf.extractall(destination)

matches = find_attached_dataset()
print('Kaggle input roots:', [str(p) for p in INPUT_ROOT.iterdir()] if INPUT_ROOT.exists() else 'missing /kaggle/input')
if not matches and INPUT_ROOT.exists():
    archives = [p for p in INPUT_ROOT.rglob('*.zip') if 'student_resource' in p.name.lower()]
    for archive in archives:
        with zipfile.ZipFile(archive) as zf:
            members = [Path(n.replace('\\', '/')).name for n in zf.namelist()]
        if not (TRAIN_REQUIRED | TEST_REQUIRED).issubset(set(members)):
            continue
        extract_root = Path('/kaggle/working/challenge_data_from_input')
        safe_extract(archive, extract_root)
        matches = find_attached_dataset()
        print('Extracted attached ZIP:', archive)
        if matches:
            break
if len(matches) != 1:
    raise RuntimeError(
        f'Expected exactly one complete attached dataset; found {len(matches)}. '
        'Use Kaggle Add Input to attach the official archive or an extracted dataset with dataset/train and dataset/test.'
    )
TRAIN_DIR, TEST_DIR = matches[0]
print('Train:', TRAIN_DIR)
print('Test: ', TEST_DIR)
for p in sorted(TRAIN_DIR.iterdir()):
    if p.is_file(): print(f'{p.name}: {p.stat().st_size / 1e9:.2f} GB')


## Select a training backend

Choose `catboost_gpu` to fit the classifier using both attached T4s, or `lightgbm_cpu` for the original V7 baseline. The candidate index and all pair features still run on CPU. A full training run is long; avoid running both backends in one session unless you intentionally want two complete runs.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

BACKEND = 'catboost_gpu'  # 'catboost_gpu' or 'lightgbm_cpu'
MAX_CANDIDATES_PER_SOURCE = 64
HARD_NEGATIVES_PER_SOURCE = 8
ROUNDS = 500

if BACKEND not in {'catboost_gpu', 'lightgbm_cpu'}:
    raise ValueError('BACKEND must be catboost_gpu or lightgbm_cpu')

if BACKEND == 'catboost_gpu':
    from catboost.utils import get_gpu_device_count
    gpu_count = get_gpu_device_count()
    print('CUDA GPUs visible to CatBoost:', gpu_count)
    if gpu_count < 1:
        raise RuntimeError('No CUDA GPU detected. Select GPU T4 x2 in Kaggle Notebook Settings and restart the session.')
    GPU_DEVICES = '0:1' if gpu_count >= 2 else '0'
    WORK_DIR = Path('/kaggle/working/v7_catboost_gpu')
    train_script = REPO_DIR / 'code/business_entity_resolution/src/train_v7_gpu.py'
    train_args = ['--task-type', 'GPU', '--devices', GPU_DEVICES, '--depth', '7', '--rounds', str(ROUNDS)]
    predict_script = REPO_DIR / 'code/business_entity_resolution/src/predict_v7_gpu.py'
else:
    GPU_DEVICES = None
    WORK_DIR = Path('/kaggle/working/v7_lightgbm_cpu')
    train_script = REPO_DIR / 'code/business_entity_resolution/src/train_v7.py'
    train_args = ['--rounds', str(ROUNDS)]
    predict_script = REPO_DIR / 'code/business_entity_resolution/src/predict_v7.py'

WORK_DIR.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, str(train_script), '--train-dir', str(TRAIN_DIR), '--work-dir', str(WORK_DIR),
       '--max-candidates-per-source', str(MAX_CANDIDATES_PER_SOURCE),
       '--hard-negatives-per-source', str(HARD_NEGATIVES_PER_SOURCE), *train_args]
print('Backend:', BACKEND, '| model devices:', GPU_DEVICES or 'CPU')
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)
report_name = 'validation_report_v7_catboost.json' if BACKEND == 'catboost_gpu' else 'validation_report_v7.json'
report_path = WORK_DIR / report_name
print(report_path.read_text() if report_path.exists() else f'Missing validation report: {report_path}')


## Full test inference and submission files

Run after the selected training cell finishes successfully. Inference uses the same saved candidate cap and source-specific thresholds. Candidate generation remains CPU-based. The outputs are written to `/kaggle/working/output_<backend>/`.

In [ ]:
OUT_DIR = Path(f'/kaggle/working/output_{BACKEND}')
OUT_DIR.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, str(predict_script), '--test-dir', str(TEST_DIR),
       '--model-dir', str(WORK_DIR), '--output-dir', str(OUT_DIR)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)
for name in ('matching_results.tsv', 'candidate_pairs.tsv'):
    path = OUT_DIR / name
    print(name, f'{path.stat().st_size / 1e6:.1f} MB' if path.exists() else 'MISSING')


In [ ]:
validator = REPO_DIR / 'utils/validate_submission.py'
cmd = [sys.executable, str(validator), '--matching', str(OUT_DIR / 'matching_results.tsv'),
       '--candidate', str(OUT_DIR / 'candidate_pairs.tsv'), '--test-dir', str(TEST_DIR)]
subprocess.run(cmd, cwd=REPO_DIR, check=True)
